<a href="https://colab.research.google.com/github/siennaLbertsch/Stock-Profile-Analyzer/blob/main/hashtable/data%20DRAFT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import yfinance as yf
import requests
from io import StringIO

stock_table = [[] for _ in range(10)]

def hash_function(ticker):
    total = 0
    for char in ticker:
        total += ord(char)
    return total % 10

def add_stock(ticker, data):
    bucket = hash_function(ticker)
    stock_table[bucket].append((ticker, data))

def get_stock(ticker):
    bucket = hash_function(ticker)
    for item in stock_table[bucket]:
        if item[0] == ticker:
            return item[1]
    return "Not found"

def update_stock(ticker, new_data):
    bucket = hash_function(ticker)
    for i, item in enumerate(stock_table[bucket]):
        if item[0] == ticker:
            stock_table[bucket][i] = (ticker, new_data)
            return
    return "Not found"

def remove_stock(ticker):
    bucket = hash_function(ticker)
    for i, item in enumerate(stock_table[bucket]):
        if item[0] == ticker:
            stock_table[bucket].pop(i)
            return
    return "Not found"


def get_sp500_metadata():
    url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
    all_tables = pd.read_html(StringIO(response.text))

    for table in all_tables:
        if "Symbol" in table.columns and "Security" in table.columns:
            sp500_table = table
            break

    meta = sp500_table[["Symbol", "Security"]].copy()
    meta.columns = ["Ticker", "Company"]
    meta["Ticker"] = meta["Ticker"].str.replace(".", "-", regex=False)

    return meta


def download_returns(meta):
    tickers = meta["Ticker"].tolist()
    data = yf.download(tickers, period="2y", interval="1d", auto_adjust=True, progress=False)

    prices = data["Close"].sort_index()
    prices = prices.dropna(axis=1, how="all")

    valid_tickers = prices.columns.tolist()
    meta = meta[meta["Ticker"].isin(valid_tickers)].reset_index(drop=True)

    returns = prices.pct_change().dropna(how="all")

    return meta, prices, returns


def main():
    print("Fetching S&P 500 company list...")
    meta = get_sp500_metadata()

    print("Downloading prices for " + str(len(meta)) + " stocks...")
    meta, prices, returns = download_returns(meta)

    for ticker in prices.columns[:10]:
        stock_info = {
            "prices": prices[ticker].dropna().tolist(),
            "returns": returns[ticker].dropna().tolist()
        }
        add_stock(ticker, stock_info)

    meta.to_csv("sp500_metadata.csv", index=False)
    prices.to_csv("sp500_prices.csv")
    returns.to_csv("sp500_returns.csv")

    print("Done! " + str(len(meta)) + " stocks saved.")
    print("\nExample lookup for AAPL:")
    result = get_stock("AAPL")
    if result != "Not found":
        print("  Latest price:  " + str(round(result["prices"][-1], 2)))
        print("  Latest return: " + str(round(result["returns"][-1], 4)))
    else:
        print(result)


main()

Fetching S&P 500 company list...


/tmp/ipykernel_7220/4245262526.py:69: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = prices.pct_change().dropna(how="all")


Done! 503 stocks saved.

Example lookup for AAPL:
  Latest price:  260.49
  Latest return: 0.0061
